In [1]:
import torch
import os
os.chdir('../')

In [2]:
from scipy import linalg
import numpy as np
import os, torch
from tqdm import tqdm
from reports.util import load_config


@torch.no_grad()
def _trace_sqrtm_product(C1: torch.Tensor, C2: torch.Tensor) -> torch.Tensor:
    # Tr sqrtm(C1 @ C2) = Tr sqrt( C1^{1/2} C2 C1^{1/2} )
    s, U = torch.linalg.eigh(C1)                 # C1 = U diag(s) U^T
    s = s.clamp_min(0)
    C1h = (U * s.sqrt()) @ U.t()                 # C1^{1/2}
    M   = C1h @ C2 @ C1h
    w   = torch.linalg.eigvalsh((M + M.t()) * 0.5).clamp_min(0)
    return w.sqrt().sum()

@torch.no_grad()
def calc_fid_stats(mu1, sigma1, mu2, sigma2, eps: float = 1e-6) -> float:
    # 모두 float64 + 동일 device로 정렬
    C1 = torch.as_tensor(sigma1, dtype=torch.float64)
    device = C1.device
    C2 = torch.as_tensor(sigma2, dtype=torch.float64).to(device)
    m1 = torch.as_tensor(mu1,    dtype=torch.float64).to(device).flatten()
    m2 = torch.as_tensor(mu2,    dtype=torch.float64).to(device).flatten()

    D = m1.numel()
    I = torch.eye(D, dtype=torch.float64, device=device)

    # 대칭화 + 정칙화
    C1 = (C1 + C1.t()) * 0.5 + eps * I
    C2 = (C2 + C2.t()) * 0.5 + eps * I

    diff = m1 - m2
    tr_covmean = _trace_sqrtm_product(C1, C2)
    fid = diff.dot(diff) + torch.trace(C1) + torch.trace(C2) - 2.0 * tr_covmean
    return float(fid)

@torch.no_grad()
def calc_fid_pt_dir(pt_dir: str, mu, sigma, eps: float = 1e-6, num=100000, key="inception_feature") -> float:
    # pt_dir에서 'inception_feature'를 모아서 mu1, sigma1 추정 후 FID 계산
    X = []
    for f in tqdm(os.listdir(pt_dir)[:num]):
        if f.endswith(".pt"):
            v = torch.load(os.path.join(pt_dir, f), map_location="cpu").get(key)
            if v is not None:
                X.append(torch.as_tensor(v, dtype=torch.float64).flatten())
    if len(X) < 2:
        raise ValueError("need >=2 features")

    X   = torch.stack(X, 0)                 # [N, D]
    mu1 = X.mean(0)
    Xc  = X - mu1
    sigma1 = (Xc.t() @ Xc) / (X.shape[0] - 1)  # 불편추정

    return calc_fid_stats(mu1, sigma1, mu, sigma, eps=eps)
    #return calculate_frechet_distance(mu1, sigma1, mu, sigma, eps=eps)

def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """Numpy implementation of the Frechet Distance.
    The Frechet distance between two multivariate Gaussians X_1 ~ N(mu_1, C_1)
    and X_2 ~ N(mu_2, C_2) is
            d^2 = ||mu_1 - mu_2||^2 + Tr(C_1 + C_2 - 2*sqrt(C_1*C_2)).

    Stable version by Dougal J. Sutherland.

    Params:
    -- mu1   : Numpy array containing the activations of a layer of the
               inception net (like returned by the function 'get_predictions')
               for generated samples.
    -- mu2   : The sample mean over activations, precalculated on an
               representative data set.
    -- sigma1: The covariance matrix over activations for generated samples.
    -- sigma2: The covariance matrix over activations, precalculated on an
               representative data set.

    Returns:
    --   : The Frechet Distance.
    """

    mu1 = np.atleast_1d(mu1)
    mu2 = np.atleast_1d(mu2)

    sigma1 = np.atleast_2d(sigma1)
    sigma2 = np.atleast_2d(sigma2)

    assert mu1.shape == mu2.shape, \
        'Training and test mean vectors have different lengths'
    assert sigma1.shape == sigma2.shape, \
        'Training and test covariances have different dimensions'

    diff = mu1 - mu2

    # Product might be almost singular
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        msg = ('fid calculation produces singular product; '
               'adding %s to diagonal of cov estimates') % eps
        print(msg)
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    # Numerical error might give slight imaginary component
    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            m = np.max(np.abs(covmean.imag))
            raise ValueError('Imaginary component {}'.format(m))
        covmean = covmean.real

    tr_covmean = np.trace(covmean)

    return (diff.dot(diff) + np.trace(sigma1)
            + np.trace(sigma2) - 2 * tr_covmean)

In [3]:
for nfe in [3, 5, 7, 9]:
    for pt_step in [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 11000, 12000, 13000, 14000, 15000, 16000, 17000, 18000, 19000, 20000]:
        pt_dir = f"samplings/GMDiT/1.4/{nfe}/Dual-Solver/10000/pt{pt_step}/traj_0"
        if not os.path.exists(pt_dir):
            continue
        
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
        print(pt_dir, fid)


100%|██████████| 10001/10001 [00:03<00:00, 2941.88it/s]


samplings/GMDiT/1.4/3/Dual-Solver/10000/pt1000/traj_0 45.62426272099714


100%|██████████| 10001/10001 [00:03<00:00, 3327.43it/s]


samplings/GMDiT/1.4/3/Dual-Solver/10000/pt2000/traj_0 48.761661327507795


100%|██████████| 10001/10001 [00:03<00:00, 3279.60it/s]


...

samplings/GMDiT/1.4/9/Dual-Solver/10000/pt18000/traj_0 4.732745855356484


100%|██████████| 10001/10001 [00:02<00:00, 3763.25it/s]


samplings/GMDiT/1.4/9/Dual-Solver/10000/pt19000/traj_0 4.734092716067266


100%|██████████| 10001/10001 [00:02<00:00, 3688.85it/s]


samplings/GMDiT/1.4/9/Dual-Solver/10000/pt20000/traj_0 4.73343430189135


In [ ]:
for nfe in [3, 5, 7, 9]:
for nfe in [9]:
    for pt_step in [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 11000, 12000, 13000, 14000, 15000, 16000, 17000, 18000, 19000, 20000]:
        pt_dir = f"samplings/GMDiT/1.4/{nfe}/Dual-Solver/10000/pt{pt_step}/mobile_0"
        if not os.path.exists(pt_dir):
            continue
        
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
        print(pt_dir, fid)


100%|██████████| 10001/10001 [00:03<00:00, 2586.32it/s]


samplings/GMDiT/1.4/9/Dual-Solver/10000/pt1000/mobile_0 4.992990048514855


100%|██████████| 10001/10001 [00:03<00:00, 2593.67it/s]


samplings/GMDiT/1.4/9/Dual-Solver/10000/pt2000/mobile_0 4.802026212753674


100%|██████████| 10001/10001 [00:04<00:00, 2498.82it/s]


...

samplings/GMDiT/1.4/9/Dual-Solver/10000/pt18000/mobile_0 4.9755050645659935


100%|██████████| 10001/10001 [00:04<00:00, 2400.83it/s]


samplings/GMDiT/1.4/9/Dual-Solver/10000/pt19000/mobile_0 4.968810813816731


100%|██████████| 10001/10001 [00:04<00:00, 2341.32it/s]


samplings/GMDiT/1.4/9/Dual-Solver/10000/pt20000/mobile_0 4.9638004590936475


In [5]:
for nfe in [3, 5, 7, 9]:
    for pt_step in [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 11000, 12000, 13000, 14000, 15000, 16000, 17000, 18000, 19000, 20000]:
        pt_dir = f"samplings/GMDiT/1.4/{nfe}/BNS-Solver/10000/pt{pt_step}/bns_0"
        if not os.path.exists(pt_dir):
            continue
        
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
        print(pt_dir, fid)


100%|██████████| 10001/10001 [00:03<00:00, 3282.96it/s]


samplings/GMDiT/1.4/3/BNS-Solver/10000/pt1000/bns_0 68.63200454126041


100%|██████████| 10001/10001 [00:03<00:00, 3075.32it/s]


samplings/GMDiT/1.4/3/BNS-Solver/10000/pt2000/bns_0 67.33378585832685


100%|██████████| 10001/10001 [00:02<00:00, 3368.25it/s]


...

samplings/GMDiT/1.4/9/BNS-Solver/10000/pt18000/bns_0 4.858815434471126


100%|██████████| 10001/10001 [00:02<00:00, 3672.69it/s]


samplings/GMDiT/1.4/9/BNS-Solver/10000/pt19000/bns_0 4.8486558699497095


100%|██████████| 10001/10001 [00:02<00:00, 3600.06it/s]


samplings/GMDiT/1.4/9/BNS-Solver/10000/pt20000/bns_0 4.859161133368048


In [8]:
for nfe in [3, 5, 7, 9]:
    for pt_step in [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 11000, 12000, 13000, 14000, 15000, 16000, 17000, 18000, 19000, 20000]:
        pt_dir = f"samplings/GMDiT/1.4/{nfe}/DS-Solver_Flow/10000/pt{pt_step}/ds_0"
        if not os.path.exists(pt_dir):
            continue
        
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
        print(pt_dir, fid)


100%|██████████| 10001/10001 [00:02<00:00, 3707.88it/s]


samplings/GMDiT/1.4/3/DS-Solver_Flow/10000/pt1000/ds_0 36.881420903714684


100%|██████████| 10001/10001 [00:02<00:00, 3809.15it/s]


samplings/GMDiT/1.4/3/DS-Solver_Flow/10000/pt2000/ds_0 37.272746305433884


100%|██████████| 10001/10001 [00:02<00:00, 3664.45it/s]


...

samplings/GMDiT/1.4/9/DS-Solver_Flow/10000/pt18000/ds_0 4.826142319559551


100%|██████████| 10001/10001 [00:02<00:00, 3762.78it/s]


samplings/GMDiT/1.4/9/DS-Solver_Flow/10000/pt19000/ds_0 4.821844619706212


100%|██████████| 10001/10001 [00:02<00:00, 3634.01it/s]


samplings/GMDiT/1.4/9/DS-Solver_Flow/10000/pt20000/ds_0 4.824827167895705


In [7]:
print('done')

done
